# Mamba SOH — LFP chemistry variant (GH-67 Mức 2)

Retrain cùng kiến trúc MambaSOHPredictor (window=30, D_MODEL=64, D_STATE=16) trên
dataset Severson et al. 2019 (Nature Energy) — LFP/graphite A123 APR18650M1A —
để có bộ artifact riêng cho chemistry LFP, KHÔNG đụng model NASA/NMC production (v1.6).

**Trước khi chạy:**
1. Settings → Accelerator → **GPU T4 x2** (KHÔNG chọn P100 — PyTorch Kaggle đã bỏ sm_60)
2. + Add Data → dataset chứa file `.mat` Severson (Batch1/2/3 từ https://data.matr.io/1/)
3. Add-ons → Secrets → `GITHUB_TOKEN` = GitHub PAT (nếu repo private)
4. **Push branch hiện tại lên GitHub trước** — Kaggle clone từ remote, không thấy local edit của bạn

Output: `soh_mamba_v2.0-lfp.pth` + `isolation_forest_v2.0-lfp.pkl` + `scaler_lfp.pkl` +
`feature_scaler_lfp.pkl` — 4 file riêng, không ghi đè lên artifact NASA (`--mamba-out`/
`--iso-out` trong scripts/train.py, thêm ở GH-67 Mức 2 đúng cho việc này).

## 1 — GPU check

In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    print('GPU:', name)
    assert 'P100' not in name, 'P100 khong tuong thich PyTorch Kaggle (sm_60) - doi sang T4 x2'
else:
    print('WARNING: bat GPU o Settings -> Accelerator -> GPU T4 x2')


## 2 — Clone repo

In [ ]:
import subprocess, os
BRANCH  = 'feat/GH-67-lfp-retrain-severson'
REPO    = '/kaggle/working/ai-module'
url = 'https://github.com/GSU26SE55/ai-module.git'
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    url = f'https://{token}@github.com/GSU26SE55/ai-module.git'
except Exception as e:
    print('No GITHUB_TOKEN secret -> thu public clone:', e)
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, url, REPO], check=True)
os.chdir(REPO)
print(subprocess.check_output(['git', 'log', '-1', '--oneline']).decode())


## 3 — Dependencies

In [ ]:
%pip install -q h5py scipy scikit-learn joblib pandas
import h5py, scipy, sklearn
print('h5py', h5py.__version__, '| scipy', scipy.__version__, '| sklearn', sklearn.__version__)


## 4 — Tìm Severson dataset

Cần thấy ít nhất 1 file `*.mat` có chữ "batch" trong tên (vd
`2017-05-12_batchdata_updated_struct_errorcorrect.mat`). Nếu assert fail —
kiểm tra lại + Add Data đã attach đúng dataset chưa.

In [ ]:
import os, subprocess
REPO = '/kaggle/working/ai-module'
found = [
    f for f in subprocess.check_output(['find', '/kaggle/input', '-iname', '*batch*.mat']).decode().splitlines()
    if f
]
assert found, 'Khong thay file *.mat co "batch" trong ten - + Add Data dataset Severson truoc'
DATASET = os.path.dirname(found[0])
os.chdir(REPO)
print('DATASET:', DATASET)
print('Files:', found)


## 5 — Preprocess (Severson .mat → window=30/6-feature)

Kiểm tra log in ra: số cell/batch, tổng số cell dùng được, và danh sách
`Val`/`Test` cell IDs (chọn ngẫu nhiên SEED=42 — xem lại nếu muốn chỉ định
tay bằng `--val-ids`/`--test-ids`).

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
!python scripts/preprocess_lfp.py --data-dir "{DATASET}" --output-dir data/processed_lfp


## 6 — Train (kiến trúc/hyperparameter GIỮ NGUYÊN — chỉ đổi data + output path)

`--mamba-out`/`--iso-out`/`--model-version` (thêm ở GH-67 Mức 2) đảm bảo KHÔNG
ghi đè `models/weights/soh_mamba_v1.6.pth` production.

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
!python scripts/train.py \
    --data-dir data/processed_lfp \
    --epochs 100 \
    --log-dir logs/training \
    --mamba-out models/weights/soh_mamba_v2.0-lfp.pth \
    --iso-out models/weights/isolation_forest_v2.0-lfp.pkl \
    --model-version 2.0-lfp \
    --feature-scaler-version 2.0-lfp


## 7 — Kiểm tra target metric

MAE < 2% / RMSE < 3% (cùng target đang áp dụng cho model NASA) — train.py tự
in WARNING nếu không đạt, cell dưới chỉ đọc lại từ checkpoint cho rõ ràng.

In [ ]:
import torch
ckpt = torch.load('models/weights/soh_mamba_v2.0-lfp.pth', map_location='cpu', weights_only=False)
print(f"Test MAE:  {ckpt['test_mae']:.4f}% (target < 2.0%)")
print(f"Test RMSE: {ckpt['test_rmse']:.4f}% (target < 3.0%)")
print('Target dat:' , ckpt['test_mae'] < 2.0 and ckpt['test_rmse'] < 3.0)


## 8 — Đóng gói artifact để tải về

4 file cần commit vào `models/weights/` (không tự commit — bạn tải zip về, tự
`git add` + `git commit` + `git push` theo quy trình repo).

In [ ]:
import shutil, os
os.makedirs('/kaggle/working/lfp_artifacts', exist_ok=True)
for p in [
    'models/weights/soh_mamba_v2.0-lfp.pth',
    'models/weights/isolation_forest_v2.0-lfp.pkl',
    'models/weights/scaler_lfp.pkl',
    'models/weights/feature_scaler_lfp.pkl',
]:
    shutil.copy(p, '/kaggle/working/lfp_artifacts/')
shutil.make_archive('/kaggle/working/lfp_artifacts', 'zip', '/kaggle/working/lfp_artifacts')
print('Tai ve: lfp_artifacts.zip (Output tab ben phai)')


## 9 — Dọn output (xoá repo clone — đã nằm trong zip)

In [ ]:
import os, shutil
os.chdir('/kaggle/working')
shutil.rmtree('/kaggle/working/ai-module', ignore_errors=True)
shutil.rmtree('/kaggle/working/lfp_artifacts', ignore_errors=True)  # da nam trong .zip
print('Output con lai:', sorted(os.listdir('/kaggle/working')))
